# SALA G14 six-source TriviaQA continuation

This notebook reuses completed generation and BLEURT artifacts from `nvthaai/multisalagithub` Version 6 and runs only projection selection, projection training, and the final BCE MLP. Use a Kaggle **GPU T4 x2** accelerator with Internet enabled.

In [ ]:
!git clone https://github.com/brainardphilemon/SALA.git /kaggle/working/SALA
%cd /kaggle/working/SALA
!git checkout 7d09466
!pip install -q -r requirements-kaggle.txt kagglehub

In [ ]:
from pathlib import Path
import kagglehub

SOURCE_HANDLE = "nvthaai/multisalagithub/versions/6"
SOURCE_NAME = "sala_qwen25_g14_6src_triviaqa_v1"
source_root = Path(kagglehub.notebook_output_download(SOURCE_HANDLE))
candidates = [source_root, *source_root.rglob(SOURCE_NAME)]
artifact_dirs = [
    path for path in candidates
    if path.name == SOURCE_NAME and (path / "generation").is_dir() and (path / "bleurt").is_dir()
]
if len(artifact_dirs) != 1:
    raise RuntimeError(f"Expected exactly one {SOURCE_NAME} artifact directory under {source_root}; found {artifact_dirs}")
ARTIFACT_DIR = artifact_dirs[0]

domains = ["tqa", "nq_open", "sciq", "popqa", "web_questions", "hotpotqa", "triviaqa"]
required = [
    *(ARTIFACT_DIR / "generation" / domain / filename for domain in domains for filename in ("rows.jsonl", "features.npy", "manifest.json")),
    *(ARTIFACT_DIR / "bleurt" / domain / filename for domain in domains for filename in ("scores.jsonl", "manifest.json")),
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise RuntimeError("Version 6 artifact download is incomplete:\n" + "\n".join(missing))
print(f"Using completed Version 6 artifacts: {ARTIFACT_DIR}")
print(f"Validated {len(required)} required generation/BLEURT files.")

In [ ]:
import subprocess
import sys

OUTPUT_DIR = Path("/kaggle/working/sala_qwen25_g14_6src_triviaqa_resume")
command = [
    sys.executable, "/kaggle/working/SALA/sala_kaggle.py",
    "--stage", "train",
    "--input-artifact-dir", str(ARTIFACT_DIR),
    "--output-dir", str(OUTPUT_DIR),
    "--settings", "G14_6SRC",
    "--samples-per-domain", "0",
    "--use-lodo-dim",
    "--classifier-loss", "bce",
]
print("Running:", " ".join(command))
subprocess.run(command, check=True)

In [ ]:
import pandas as pd

results = pd.read_csv(OUTPUT_DIR / "results.csv")
display(results.head())
print(f"Rows: {len(results):,}")
print(f"Saved to: {OUTPUT_DIR}")